# Machine Learning as Function Finding
## Model, Loss, Optimization, and Generalization

At its core, **machine learning is the task of automatically finding a function `f`** that maps inputs to useful outputs. Instead of hand-coding rules, we let an algorithm search for a good `f` from data.

This search decomposes into **three steps**:

1. **Model** — choose a *set* of candidate functions `H` (the hypothesis set). Picking a model means picking the shape of the functions we are willing to consider.
2. **Loss** — define a goodness criterion `L(f)` that measures how badly a candidate function fits the data.
3. **Optimization** — find the best function `f* = argmin_{f \in H} L(f)`.

### Roadmap of this notebook

- **Build the framework** on a simple linear-regression example: define a model, define a loss, and *feel* the loss by moving sliders (C03–C06).
- **Learn how optimization works** with gradient descent, and watch how the **learning rate** controls convergence, oscillation, and divergence (C07–C09).
- **See why low training loss is not enough**: a model can memorize the training data yet fail on new data. We explore **overfitting** and how **regularization** helps (C10–C13).

Everything runs on small synthetic datasets generated inside the notebook — no downloads, no GPU. Run the cells top to bottom.

In [ ]:
# --- Setup: imports, plotting config, seeded RNG, and a styling helper ---
import numpy as np
import matplotlib.pyplot as plt

# Enable inline plotting when running inside IPython/Colab; ignore otherwise.
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

# ipywidgets is optional: if missing, later cells fall back to static plots.
try:
    from ipywidgets import interact, FloatSlider, IntSlider, FloatLogSlider
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
    interact = FloatSlider = IntSlider = FloatLogSlider = None
    print("ipywidgets not found -> interactive cells will render a static plot.")
    print("To enable sliders, run:  !pip install ipywidgets")

# Reproducibility: one shared seeded generator used by every data-generating cell.
SEED = 42
rng = np.random.default_rng(SEED)

# Consistent figure styling reused across the notebook.
plt.rcParams['figure.figsize'] = (8, 4.5)
plt.rcParams['figure.dpi'] = 110


def style_axes(ax, title=None, xlabel='x', ylabel='y'):
    """Apply consistent labels, optional title, and a light grid to an Axes."""
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title is not None:
        ax.set_title(title)
    ax.grid(alpha=0.3)
    return ax


# Sanity checks
assert 'np' in dir() and 'plt' in dir()
assert isinstance(rng, np.random.Generator)

print(f"Environment ready | numpy {np.__version__} | matplotlib {plt.matplotlib.__version__} | seed={SEED} | widgets={WIDGETS_AVAILABLE}")

## Step 1 — Model: choosing the set of candidate functions `H`

The **model** is the *set* of functions we allow ourselves to search over — the **hypothesis set** `H`. Choosing a model means choosing the *shape* of `f` before we ever look at the loss.

The simplest useful `H` is the set of **straight lines**:

$$f(x) = w\,x + b$$

Each choice of the parameters `(w, b)` picks **one** function out of this infinite set. Learning will later mean searching over `(w, b)` for the best line.

### A taxonomy by output type

Models are often grouped by *what kind of output* `f` produces:

| Type | Output | Example |
|------|--------|---------|
| **Regression** | a continuous number | predict tomorrow's temperature |
| **Classification** | a category / label | spam vs. not-spam |
| **Generative** | structured content | generate an image or a sentence |

In this notebook we focus on **regression** with the linear model above, because it makes the three-step framework concrete and easy to visualize. The same Model → Loss → Optimization recipe applies to all three types.

In [ ]:
# --- Generate the synthetic 1-D linear dataset (reused by C06, C08, C09) ---
true_w, true_b = 2.0, -1.0   # the unknown line we will try to recover
N = 40                       # number of training points
noise_std = 1.0              # observation noise

# Sort x so the candidate line plots cleanly left-to-right.
x_train = np.sort(rng.uniform(-3, 3, size=N))
y_train = true_w * x_train + true_b + rng.normal(0, noise_std, size=N)

assert x_train.shape == y_train.shape == (N,)
assert np.all(np.diff(x_train) >= 0)  # x_train is sorted

# One example candidate line from the hypothesis set H (a poor initial guess).
w_guess, b_guess = 1.0, 0.0
xs = np.linspace(x_train.min(), x_train.max(), 100)

fig, ax = plt.subplots()
ax.scatter(x_train, y_train, color='tab:blue', alpha=0.8, label='training data')
ax.plot(xs, w_guess * xs + b_guess, color='tab:red', lw=2,
        label=f'candidate f(x) = {w_guess}x + {b_guess}')
style_axes(ax, title='A synthetic linear dataset and one candidate function')
ax.legend()
plt.show()

print(f"Generated {N} points from true line: y = {true_w}*x + ({true_b}) + noise(std={noise_std})")

## Step 2 — Loss: how good is a candidate function?

Given a candidate `f`, the **loss** `L(f)` measures how far its predictions land from the true labels. The smaller the loss, the better the function fits the data.

A natural choice is to **add up the distances** between each prediction `f(xᵢ)` and its label `yᵢ`. Squaring those distances and averaging gives the **Mean Squared Error (MSE)**:

$$L(w, b) = \frac{1}{N}\sum_{i=1}^{N} \big(f(x_i) - y_i\big)^2 = \frac{1}{N}\sum_{i=1}^{N} \big(w\,x_i + b - y_i\big)^2$$

Key points:

- Loss is a function **of the parameters** `(w, b)` — every candidate line gets a single score.
- MSE is always `>= 0`; it is `0` only if the line passes exactly through every point.
- Squaring penalizes large errors more than small ones and keeps the loss smooth (handy for optimization later).

This is the **supervised** setting: every input `xᵢ` has a known label `yᵢ`. (In *semi-supervised* learning only some points are labeled, and the criterion is adjusted accordingly.)

Next, let's compute this loss interactively and *feel* how `(w, b)` changes a good fit into a bad one.

In [ ]:
# --- Define the MSE loss and explore it interactively ---
def loss(w, b):
    """Mean squared error of the line f(x)=w*x+b over the training data."""
    preds = w * x_train + b
    return float(np.mean((preds - y_train) ** 2))

# Fixed axis limits so the view does not jump while sliders move.
_pad = 1.0
_xlim = (x_train.min() - 0.2, x_train.max() + 0.2)
_ylim = (y_train.min() - _pad, y_train.max() + _pad)
_xs = np.linspace(x_train.min(), x_train.max(), 100)


def plot_line(w, b):
    fig, ax = plt.subplots()
    ax.scatter(x_train, y_train, color='tab:blue', alpha=0.8, label='data')
    ax.plot(_xs, w * _xs + b, color='tab:red', lw=2, label='candidate line')
    ax.set_xlim(_xlim)
    ax.set_ylim(_ylim)
    style_axes(ax, title=f'Loss  L(w={w:.2f}, b={b:.2f}) = {loss(w, b):.3f}')
    ax.legend(loc='upper left')
    plt.show()


# Sanity checks
assert loss(true_w, true_b) >= 0.0
assert loss(true_w, true_b) < loss(0.0, 0.0)  # the truth fits better than a flat zero line

if WIDGETS_AVAILABLE:
    interact(plot_line,
             w=FloatSlider(min=-1, max=5, step=0.1, value=1.0, description='w'),
             b=FloatSlider(min=-5, max=5, step=0.1, value=0.0, description='b'))
else:
    print("Static fallback (widgets unavailable): showing the best-fit line.")
    plot_line(true_w, true_b)

## Step 3 — Optimization: finding the best function

We now have a model `H` and a loss `L`. Optimization is the search for the best member of `H`:

$$f^{*} = \arg\min_{f \in H} L(f) \qquad\Longleftrightarrow\qquad (w^{*}, b^{*}) = \arg\min_{w, b} L(w, b)$$

For smooth losses we can use **gradient descent**: start somewhere, then repeatedly step *downhill* along the negative gradient.

$$w \leftarrow w - \eta \,\frac{\partial L}{\partial w}, \qquad b \leftarrow b - \eta \,\frac{\partial L}{\partial b}$$

For MSE the gradients are analytic. With `err = (w·xᵢ + b) − yᵢ`:

$$\frac{\partial L}{\partial w} = \frac{2}{N}\sum_i err_i\, x_i, \qquad \frac{\partial L}{\partial b} = \frac{2}{N}\sum_i err_i$$

### The learning rate `η`

The step size `η` (eta) is a **hyperparameter** — a knob we set *before* training begins, not something the algorithm learns. It is decisive:

- **Too small** → painfully slow convergence.
- **Just right** → smooth, steady descent to the minimum.
- **Too large** → overshooting, oscillation, or outright divergence.

Let's implement gradient descent and then make the learning rate's effect tangible.

In [ ]:
# --- Gradient descent for MSE (analytic gradients); reused by C09 ---
def gradient_descent(lr, n_steps, w0=0.0, b0=0.0):
    """Run GD on the MSE loss. Returns the trajectory as arrays of w, b, loss.

    Includes the initial point. Stops early (returning the partial trajectory)
    if the loss becomes non-finite, so divergence can still be visualized.
    """
    n_steps = max(1, int(n_steps))
    w, b = float(w0), float(b0)
    ws, bs, losses = [w], [b], [loss(w, b)]
    for _ in range(n_steps):
        preds = w * x_train + b
        err = preds - y_train
        grad_w = (2.0 / N) * np.sum(err * x_train)
        grad_b = (2.0 / N) * np.sum(err)
        w -= lr * grad_w
        b -= lr * grad_b
        cur = loss(w, b)
        ws.append(w); bs.append(b); losses.append(cur)
        if not np.isfinite(cur):  # numeric blow-up -> stop, keep partial path
            break
    return {'w': np.asarray(ws, dtype=float),
            'b': np.asarray(bs, dtype=float),
            'loss': np.asarray(losses, dtype=float)}


# Sanity checks
traj = gradient_descent(0.05, 200)
assert traj['loss'][-1] < traj['loss'][0]
assert abs(traj['w'][-1] - true_w) < 0.5 and abs(traj['b'][-1] - true_b) < 0.5
assert len(traj['w']) == len(traj['loss'])

# Example run for the reader
example = gradient_descent(lr=0.05, n_steps=50)
print(f"After 50 steps (lr=0.05): w={example['w'][-1]:.3f}, b={example['b'][-1]:.3f}, "
      f"loss={example['loss'][-1]:.3f}")
print(f"True parameters:          w={true_w:.3f}, b={true_b:.3f}, "
      f"loss={loss(true_w, true_b):.3f}")

In [ ]:
# --- Interactive gradient-descent visualizer: learning rate & steps ---
# Precompute the loss surface once (kept at module scope for responsiveness).
W = np.linspace(-1, 5, 80)
B = np.linspace(-5, 5, 80)
WW, BB = np.meshgrid(W, B)
ZZ = np.empty_like(WW)
for i in range(WW.shape[0]):
    for j in range(WW.shape[1]):
        ZZ[i, j] = loss(WW[i, j], BB[i, j])
assert ZZ.shape == WW.shape == BB.shape


def show_descent(lr, n_steps):
    t = gradient_descent(lr, n_steps)
    finite = np.isfinite(t['w']) & np.isfinite(t['b']) & np.isfinite(t['loss'])
    diverged = not np.all(finite)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.8))

    # Left: loss-surface contour with the descent path overlaid.
    cf = ax1.contourf(WW, BB, ZZ, levels=30, cmap='viridis')
    ax1.contour(WW, BB, ZZ, levels=12, colors='white', alpha=0.3, linewidths=0.6)
    fig.colorbar(cf, ax=ax1, label='loss')
    ax1.plot(t['w'][finite], t['b'][finite], '-o', color='tab:red',
             ms=3, lw=1.2, label='descent path')
    ax1.scatter([true_w], [true_b], marker='*', s=220, color='gold',
                edgecolor='black', zorder=5, label='true (w, b)')
    ax1.set_xlim(W.min(), W.max())
    ax1.set_ylim(B.min(), B.max())
    style_axes(ax1, title='Loss surface & descent path', xlabel='w', ylabel='b')
    ax1.legend(loc='upper right')

    # Right: loss-vs-iteration curve.
    it = np.arange(len(t['loss']))
    pos_finite = finite & (t['loss'] > 0)
    if np.all(finite) and np.all(t['loss'] > 0):
        ax2.semilogy(it, t['loss'], '-o', color='tab:blue', ms=3)
    else:
        ax2.plot(it[pos_finite], t['loss'][pos_finite], '-o', color='tab:blue', ms=3)
    final = t['loss'][-1]
    title = ('diverged (try a smaller learning rate)' if diverged
             else f'final loss = {final:.3f}')
    style_axes(ax2, title=title, xlabel='iteration', ylabel='loss')

    fig.suptitle(f'lr = {lr:.4g},  steps = {int(n_steps)}')
    fig.tight_layout()
    plt.show()


# Sanity checks
show_descent(0.05, 50)   # convergent case runs without error
show_descent(1.5, 50)    # large lr -> diverges, still runs without error

if WIDGETS_AVAILABLE:
    interact(show_descent,
             lr=FloatLogSlider(base=10, min=-3, max=0.5, step=0.1, value=0.05,
                               description='learning rate'),
             n_steps=IntSlider(min=5, max=300, step=5, value=50, description='steps'))
else:
    print("Static fallback (widgets unavailable): convergent run shown above.")

## From training loss to generalization

Gradient descent can drive the **training loss** very low. But a tiny training loss is *not* the real goal — we want the function to work on **new, unseen data**.

Imagine a student who memorizes the exact answers to last year's exam. They score perfectly on those questions but fail when the test changes. A model can do exactly the same thing: **overfitting** is fitting the training data so closely that it captures the *noise* instead of the underlying pattern, and then performs poorly on new inputs.

To detect this we split the data:

- **Training set** — used to fit the model.
- **Test set** — held out, used only to estimate performance on unseen data.

We also need a knob for **model capacity** — how flexible `H` is. For polynomials,

$$f(x) = c_0 + c_1 x + c_2 x^2 + \dots + c_d x^d,$$

the **degree `d`** controls capacity: higher degree = larger, more flexible `H`. Low degree may **underfit** (too rigid to capture the pattern); very high degree tends to **overfit** (flexible enough to chase noise).

In the next cells we build a nonlinear dataset, watch test error rise as degree grows, and then use **regularization** to push back.

In [ ]:
# --- Nonlinear dataset for the overfitting demo, split into train/test ---
def g(x):
    """The (unknown) true function we are trying to recover."""
    return np.sin(1.5 * x)

M = 30
x_all = np.sort(rng.uniform(-3, 3, size=M))
y_all = g(x_all) + rng.normal(0, 0.25, size=M)

# Shuffle indices, take ~60% for training.
idx = rng.permutation(M)
n_tr = int(round(0.6 * M))
tr_idx = np.sort(idx[:n_tr])
te_idx = np.sort(idx[n_tr:])

x_train2, y_train2 = x_all[tr_idx], y_all[tr_idx]
x_test2, y_test2 = x_all[te_idx], y_all[te_idx]

assert len(x_train2) + len(x_test2) == M
assert len(x_train2) > 0 and len(x_test2) > 0
assert len(np.intersect1d(tr_idx, te_idx)) == 0  # disjoint

# Plot the splits with the true function overlaid.
xs2 = np.linspace(-3, 3, 300)
fig, ax = plt.subplots()
ax.plot(xs2, g(xs2), 'k--', alpha=0.5, lw=1.5, label='true function g(x)=sin(1.5x)')
ax.scatter(x_train2, y_train2, color='tab:blue', s=45, label='train')
ax.scatter(x_test2, y_test2, color='tab:orange', marker='^', s=55, label='test')
style_axes(ax, title=f'Nonlinear data: {len(x_train2)} train / {len(x_test2)} test points')
ax.legend()
plt.show()

In [ ]:
# --- Interactive polynomial fitting: watch overfitting appear ---
import warnings


def mse(y_true, y_pred):
    """Mean squared error helper (reused by C13)."""
    return float(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))


# Bound the view so exploding high-degree fits stay readable.
_y2_pad = 1.0
_y2lim = (min(y_train2.min(), y_test2.min()) - _y2_pad,
          max(y_train2.max(), y_test2.max()) + _y2_pad)
_xs2 = np.linspace(-3, 3, 300)


def plot_poly(degree):
    degree = int(degree)
    degree = min(degree, len(x_train2) - 1)  # cannot exceed #train-1
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', np.RankWarning)
        try:
            coeffs = np.polyfit(x_train2, y_train2, degree)
        except np.linalg.LinAlgError:
            print(f"polyfit failed at degree {degree}")
            return
    tr = mse(y_train2, np.polyval(coeffs, x_train2))
    te = mse(y_test2, np.polyval(coeffs, x_test2))

    fig, ax = plt.subplots()
    ax.plot(_xs2, np.polyval(coeffs, _xs2), color='tab:red', lw=2, label='polynomial fit')
    ax.scatter(x_train2, y_train2, color='tab:blue', s=45, label='train')
    ax.scatter(x_test2, y_test2, color='tab:orange', marker='^', s=55, label='test')
    ax.set_ylim(_y2lim)
    style_axes(ax, title=f'degree={degree}   train MSE={tr:.3f}   test MSE={te:.3f}')
    ax.legend(loc='upper right')
    plt.show()


# Sanity checks: higher degree never increases training error
tr1 = mse(y_train2, np.polyval(np.polyfit(x_train2, y_train2, 1), x_train2))
tr9 = mse(y_train2, np.polyval(np.polyfit(x_train2, y_train2, 9), x_train2))
assert tr9 <= tr1 + 1e-9

if WIDGETS_AVAILABLE:
    interact(plot_poly, degree=IntSlider(min=1, max=15, step=1, value=1, description='degree'))
else:
    print("Static fallback (widgets unavailable): high-degree fit (degree=9).")
    plot_poly(9)

In [ ]:
# --- Regularization (ridge) as a remedy for overfitting ---
DEG = 12  # deliberately high capacity


def ridge_fit(X, y, lam):
    """Closed-form ridge regression. Returns theta. The intercept column
    (last column of np.vander) is NOT penalized."""
    n_features = X.shape[1]
    I = np.eye(n_features)
    I[-1, -1] = 0.0  # do not shrink the bias term
    A = X.T @ X + lam * I
    rhs = X.T @ y
    try:
        theta = np.linalg.solve(A, rhs)
    except np.linalg.LinAlgError:
        theta, *_ = np.linalg.lstsq(A, rhs, rcond=None)
    return theta


_Xtr = np.vander(x_train2, DEG + 1)
_Xte = np.vander(x_test2, DEG + 1)
_Xgrid = np.vander(_xs2, DEG + 1)


def plot_ridge(log_lambda):
    lam = 10.0 ** log_lambda
    theta = ridge_fit(_Xtr, y_train2, lam)
    tr = mse(y_train2, _Xtr @ theta)
    te = mse(y_test2, _Xte @ theta)

    fig, ax = plt.subplots()
    ax.plot(_xs2, _Xgrid @ theta, color='tab:green', lw=2, label=f'ridge fit (deg={DEG})')
    ax.scatter(x_train2, y_train2, color='tab:blue', s=45, label='train')
    ax.scatter(x_test2, y_test2, color='tab:orange', marker='^', s=55, label='test')
    ax.set_ylim(_y2lim)
    style_axes(ax, title=f'lambda={lam:.1e}   train MSE={tr:.3f}   test MSE={te:.3f}')
    ax.legend(loc='upper right')
    plt.show()


# Sanity check: a larger penalty shrinks the coefficient norm.
_t_small = ridge_fit(_Xtr, y_train2, 1e-6)
_t_big = ridge_fit(_Xtr, y_train2, 1e3)
assert np.linalg.norm(_t_big) < np.linalg.norm(_t_small)

if WIDGETS_AVAILABLE:
    interact(plot_ridge,
             log_lambda=FloatSlider(min=-6, max=3, step=0.5, value=-3,
                                    description='log10(lambda)'))
else:
    print("Static fallback (widgets unavailable): moderate regularization (lambda=1e-3).")
    plot_ridge(-3)

## Recap & where to go next

We framed machine learning as **automatically finding a function `f`**, built from three steps:

1. **Model** — choose the hypothesis set `H` (lines, then polynomials).
2. **Loss** — score each candidate with `L(f)` (mean squared error).
3. **Optimization** — find `f* = argmin L(f)` (gradient descent).

What each demo showed:

- **C06 — the loss landscape.** Sliding `(w, b)` made `L(w, b)` tangible: good functions sit in the low-loss valley.
- **C09 — gradient descent & the learning rate.** The same valley seen as a contour; the step size `η` decided whether we converged smoothly, oscillated, or diverged.
- **C12 — overfitting.** Increasing polynomial degree drove **training** error toward zero while **test** error climbed — the model memorized noise.
- **C13 — regularization.** An L2 (ridge) penalty tamed the wild high-degree fit and narrowed the train/test gap.

### Extensions to try

- Swap plain gradient descent for **momentum or Adam** and compare convergence on the C09 surface.
- Add **more input features** (multivariate regression) and inspect the loss in higher dimensions.
- Move from regression to **classification** (e.g. logistic regression with cross-entropy loss) — same three-step recipe, different `H` and `L`.
- Use **cross-validation** to choose the polynomial degree or `lambda` instead of eyeballing the test curve.

The recipe — *Model → Loss → Optimization*, watched through the lens of *generalization* — scales from this notebook all the way up to modern deep learning.